# Stage 4 (part 1) — Shared runner, linear models, tree ensembles

| Part | Prompt | Code |
|---|---|---|
| 1 | 4.1 shared experiment runner | `src/experiment.py`, `tests/test_experiment.py` |
| 2 | 4.2 linear family | `src/models.py` (`LINEAR_CLS`, `LINEAR_REG`) |
| 3 | 4.3 Random Forest, XGBoost, LightGBM, CatBoost | `src/models.py` (`trees`) |

Pre-registered metrics (Decision 3.7 / P1, approved): **classification** — BSS vs climatology (headline), PR-AUC,
ROC-AUC, CSI at a frozen threshold; **regression** — MAE (headline), MSESS, RMSE, wet-day MAE. Following P3 we also report
skill vs **seasonal + persistence** and folds 1–3 vs 4–5. Rule-11 triggers (P2): BSS > 0.45, monsoon ROC-AUC > 0.90,
MSESS > 0.60. Classification uses the primary target (≥ 1 mm); 0.1/2.5 mm follow for the final models.
The test set is not touched.

### Step 1 — Setup

**WHAT:** imports, paths, seeds; import the shared modules.

**WHY:** every model in this notebook goes through the same runner (rule 7), so all differences come from the models.

**Look for:** library versions (rule 13: APIs were checked against these versions).

In [ ]:
import sys, subprocess, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, xgboost, lightgbm, catboost
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, lasso_path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import experiment as E, models as Mo, splits as S

FIG, TAB = ROOT / "reports/figures", ROOT / "reports/tables"
np.random.seed(0)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 3)
def save(fig, name): fig.savefig(FIG / name, dpi=300, bbox_inches="tight")
print("sklearn", sklearn.__version__, "| xgboost", xgboost.__version__, "| lightgbm", lightgbm.__version__,
      "| catboost", catboost.__version__, "| seeds", Mo.SEEDS)

<<A1>>

# Part 1 — Prompt 4.1: the shared runner

### Step 2 — Design of `run_cv` (and why)

For each of the 5 folds:

1. **Inner split.** The training fold is split into *inner-train* and a **"slice"** = its last 3 target years
   (e.g. fold 1: inner-train 1981–2001, slice 2002–2004), with the same 1-day purge.
2. **Inner fit → choices.** A copy of the model is fitted on inner-train and predicts the slice. From these
   out-of-sample predictions we take (a) the **decision threshold** that maximises CSI and (b) for gradient boosters the
   **number of trees** at which the slice loss stops improving (early stopping, patience 100).
3. **Refit.** A fresh model is fitted on the **whole training fold** (boosters with the number of trees from step 2), so the
   most recent years are used for learning, then it predicts the validation block.
4. **Score & store.** Validation predictions are scored with `src/metrics.py`; skill references (climatology,
   seasonal + persistence) are the Stage-3 baselines fitted on the same training fold. Out-of-fold (OOF) predictions →
   `models/oof/<model>__<task>__<set>__seed<k>.parquet`; tidy rows `(model, task, feature_set, fold, seed, metric, value)` →
   `reports/tables/cv_results.csv`. Re-running a configuration **replaces** its old rows (no duplicates).

**Design choices and alternatives**

| choice | why | alternative considered |
|---|---|---|
| threshold & early stopping on the last 3 training years | out-of-sample for the inner model, never touches the validation block (your Stage-3 answer 3); most recent years resemble the validation period best | (a) the validation block itself → optimistic, peeks at labels; (b) in-sample training rows → over-confident ML probabilities; (c) earlier folds' OOF → fold 1 would have none |
| refit on the full training fold | keeps the newest 3 years for learning (important with the regime shifts) | keep the inner model → loses the most relevant data |
| keep the inner number of trees unchanged at refit | simple, common practice; the full fold is only ~12–25 % larger | scale trees by data size → an extra assumption |
| Pipelines (one-hot month + StandardScaler) for linear models | rule 6: preprocessing fitted on training rows only | scaling the whole table once → leaks validation means/stds |
| raw features for trees | splits are invariant to monotone rescaling | — |
| deterministic models 1 seed, stochastic 5 seeds | rule 8; repeating a deterministic fit gives identical numbers | 5 seeds for all → wasted compute |
| regression trained on mm (`y_reg`), predictions clipped at 0 | MAE/RMSE are evaluated in mm; negative rain is impossible | `y_reg_log` (Jensen bias, Stage 2 Q3) → part 2 |
| `class_weight="balanced"` | requested; with ~49 % rain days it changes weights by only ~±5 % | none — note it can distort probabilities when classes are imbalanced |

**Approximation to keep in mind:** the threshold is chosen for the inner model and applied to the refitted model; their
probabilities are very similar but not identical.

**Look for (next step):** all runner tests pass.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(ROOT / "tests")],
                   capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-600:], r.stderr[-300:])

<<A2>>

### Step 3 — Plumbing check with trivial models, and baseline rows in the same table

**WHAT:** (i) write the Stage-3 baselines into `cv_results.csv` with exactly the same metric code (`baseline:<name>`,
feature set "-"); (ii) run a trivial model through the runner: `DummyClassifier(strategy="prior")` (predicts the training
base rate) and `DummyRegressor()` (training mean), on Set A.

**WHY:** a trivial model has known answers: ROC-AUC = 0.5, and its scores must equal the majority/mean baseline exactly,
because it *is* the same forecast computed through a different code path. Agreement proves the runner's splits, metric
calls and storage are wired correctly.

**Alternatives considered:** going straight to a real model — a plumbing error would then be indistinguishable from a
modelling result.

**Look for:** dummy = majority on every fold (difference 0), ROC-AUC 0.5.

In [ ]:
for task in ("cls", "reg"):
    E.run_baselines(task)
dummy = {"cls": lambda s, y: DummyClassifier(strategy="prior"), "reg": lambda s, y: DummyRegressor()}
check = {}
for task, metric in (("cls", "BSS"), ("reg", "MAE")):
    res_d, oof_d, log_d = E.run_cv("dummy", dummy[task], task, "A")
    cv = pd.read_csv(E.RESULTS)
    base = cv[(cv.model == "baseline:majority") & (cv.task == task) & (cv.metric == metric)].sort_values("fold")["value"].to_numpy()
    mine = res_d[res_d.metric == metric].sort_values("fold")["value"].to_numpy()
    check[f"{task}: {metric} dummy - majority (max abs diff)"] = np.abs(mine - base).max()
check["cls: dummy ROC-AUC (all folds)"] = pd.read_csv(E.RESULTS).query("model=='dummy' and task=='cls' and metric=='ROC_AUC'")["value"].unique()
pd.Series(check, dtype=object).to_frame("value")

<<A3>>

### Step 4 — Sanity check of one fold: the fold log and the out-of-fold predictions

**WHAT:** show the inner split / threshold log of the dummy classifier, and the first OOF rows of fold 5.

**WHY:** to see with our own eyes that the slice lies inside the training fold, the threshold was chosen there, and the
stored predictions belong to the validation days.

**Look for:** slices ending in 2004, 2007, …, 2016; OOF index starting 2016-12-31 (target 2017-01-01) for fold 5.

In [ ]:
_, oof_d, log_d = E.run_cv("dummy", dummy["cls"], "cls", "A", save=False)
display(log_d)
oof_d[oof_d.fold == 5].head()

<<A4>>

# Part 2 — Prompt 4.2: the linear family

### Step 5 — Models

| task | model | penalty (scikit-learn 1.9 API) | solver | seeds |
|---|---|---|---|---|
| cls | `logreg_none` | none (`C=inf`) | lbfgs | 1 |
| cls | `logreg_l2` | L2, `C=1` (`l1_ratio=0`) | lbfgs | 1 |
| cls | `logreg_l1` | L1, `C=0.1` (`l1_ratio=1`) | saga (stochastic) | 5 |
| cls | `logreg_enet` | elastic net, `C=0.1`, `l1_ratio=0.5` | saga | 5 |
| cls | `sgd_logloss` | L2, `alpha=1e-4`, log loss | SGD | 5 |
| reg | `linreg` | none | least squares | 1 |
| reg | `ridge` | L2, `alpha=1` | closed form | 1 |
| reg | `lasso` | L1, `alpha=0.1` | coordinate descent | 1 |
| reg | `elasticnet` | `alpha=0.1`, `l1_ratio=0.5` | coordinate descent | 1 |
| reg | `sgd_reg` | L2, `alpha=1e-4`, squared loss | SGD | 5 |

All classifiers use `class_weight="balanced"`. All inputs: one-hot month (drop first) + standardised features inside the
Pipeline. **Not tuned.** The only non-default values are `C=0.1` for the L1/elastic-net logistic models and
`alpha=0.1` for Lasso/ElasticNet, chosen *a priori* (not by looking at scores) so that the sparsity is visible; with the
defaults (`C=1`, `alpha=1`) L1 logistic keeps almost all features and Lasso keeps very few at this rainfall scale (mm).

**Rule 13 note:** in scikit-learn 1.9 `LogisticRegression(penalty=...)` is deprecated; the penalty is set by `C` and
`l1_ratio` (verified on the installed version).

**Look for:** — (definitions)

### Step 6 — Run all linear models (Sets A and B)

**WHAT:** every model × feature set × seed through `run_cv`. One printed line per configuration.

**WHY:** identical folds, thresholds procedure, metrics and storage for all (rule 7).

**Look for:** no warnings about non-convergence; run times.

In [ ]:
def run_family(specs, task, fsets=("A", "B")):
    for fset in fsets:
        for name, (factory, stochastic) in specs.items():
            for seed in (Mo.SEEDS if stochastic else [0]):
                t0 = time.time()
                with warnings.catch_warnings(record=True) as w:
                    warnings.simplefilter("always")
                    res_k, _, _ = E.run_cv(name, factory, task, fset, seed)
                head = res_k[res_k.metric == ("BSS" if task == "cls" else "MAE")]["value"].mean()
                print(f"{task} {fset} {name:13s} seed {seed}: {'BSS' if task == 'cls' else 'MAE'} {head:.3f} "
                      f"({time.time() - t0:5.1f}s){'  WARN: ' + str({x.category.__name__ for x in w}) if w else ''}")

run_family(Mo.LINEAR_CLS, "cls")
run_family(Mo.LINEAR_REG, "reg")

<<A5>>

### Step 7 — Linear models vs baselines (mean ± fold std; folds 1–3 vs 4–5; seed std)

**WHAT:** the pre-registered metrics for every linear model and the main baselines. `mean ± std` = mean over the 5 folds
(seed-averaged) ± std across folds; `folds 1–3` / `folds 4–5` = means before / across the 2016 jump; `seed std` = std of the
5-fold mean across seeds (0 for deterministic models).

**WHY:** P1 and P3. The skill vs seasonal + persistence column answers "did the model learn anything beyond season and
persistence?"

**Look for:** BSS above 0.184 and BSS_vs_seas_pers > 0; regression MAE below 7.08 and MSESS_vs_seas_pers > 0; whether gains
come from folds 4–5 only.

In [ ]:
CLS_KEY = ["BSS", "BSS_vs_seas_pers", "PR_AUC", "ROC_AUC", "ROC_AUC_monsoon", "CSI", "ECE"]
REG_KEY = ["MAE", "MSESS", "MSESS_vs_seas_pers", "RMSE", "MAE_wet", "bias", "recall_ge20"]
BASE = ["baseline:seasonal+persistence", "baseline:climatology", "baseline:persistence"]

def compare(task, models, keys, headline, name):
    cv = pd.read_csv(E.RESULTS)
    s = E.summarise(cv[(cv.task == task) & cv.model.isin(models)], keys)
    s["cell"] = s["mean"].map("{:.3f}".format) + " ± " + s["fold_std"].map("{:.3f}".format)
    wide = s.pivot_table(index=["model", "feature_set"], columns="metric", values="cell", aggfunc="first")[keys]
    h = s[s.metric == headline].set_index(["model", "feature_set"])
    wide[f"{headline} folds 1-3"] = h["folds_1_3"].round(3)
    wide[f"{headline} folds 4-5"] = h["folds_4_5"].round(3)
    wide[f"{headline} seed std"] = h["seed_std"].fillna(0).round(4)
    order = h["mean"].sort_values(ascending=(headline == "MAE")).index
    wide = wide.loc[order]
    wide.to_csv(TAB / f"{name}.csv")
    return wide

display(compare("cls", BASE + list(Mo.LINEAR_CLS), CLS_KEY, "BSS", "04_linear_vs_baselines_cls"))
compare("reg", BASE + list(Mo.LINEAR_REG), REG_KEY, "MAE", "04_linear_vs_baselines_reg")

<<A6>>

### Step 8 — Rule-11 check

**WHAT:** list any fold of any model where BSS > 0.45, monsoon ROC-AUC > 0.90 or MSESS > 0.60 (approved triggers, P2).

**WHY:** suspiciously good scores must be investigated before they are reported.

**Look for:** an empty table.

In [ ]:
def rule11():
    cv = pd.read_csv(E.RESULTS)
    trig = {"BSS": 0.45, "ROC_AUC_monsoon": 0.90, "MSESS": 0.60}
    hits = cv[cv.metric.isin(trig) & (cv.value > cv.metric.map(trig))]
    print(f"configurations checked: {cv.groupby(E.KEYS).ngroups}; fold-level trigger hits: {len(hits)}")
    print("max per metric:", cv[cv.metric.isin(trig)].groupby("metric")["value"].max().round(3).to_dict())
    return hits
rule11()

<<A7>>

### Step 9 — Sanity look at predictions: fold 5, one wet season

**WHAT:** observed rain and the out-of-fold predictions of `logreg_l2` (probability) and `ridge` (mm) for March–October
2018 (Set B), next to the seasonal + persistence baseline.

**WHY:** numbers can hide silly behaviour (e.g. a constant forecast, a one-day shift). Eyeballing a season checks that
probabilities rise before and during rain and that mm forecasts are plausible.

**Look for:** probabilities near 1 in the monsoon and dropping in dry breaks; regression forecasts much smoother than
observations (they cannot forecast 100-mm days).

In [ ]:
o_c = pd.read_parquet(E.OOF_DIR / "logreg_l2__cls__B__seed0.parquet")
o_r = pd.read_parquet(E.OOF_DIR / "ridge__reg__B__seed0.parquet")
win = slice("2018-03-01", "2018-10-31")
t = S.target_dates(o_c.loc[win].index)
fig, (a0, a1) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
a0.bar(t, o_c.loc[win, "y_true"], color="lightsteelblue", width=1, label="observed rain >= 1 mm")
a0.plot(t, o_c.loc[win, "pred"], color="tab:red", lw=1, label="logreg_l2 P(rain)")
a0.axhline(o_c.loc[win, "threshold"].iloc[0], color="k", ls=":", lw=1, label="frozen threshold")
a0.legend(fontsize=7, loc="lower center"); a0.set_ylabel("probability")
a1.bar(t, o_r.loc[win, "y_true"], color="lightsteelblue", width=1, label="observed mm")
a1.plot(t, o_r.loc[win, "pred"], color="tab:red", lw=1, label="ridge forecast")
a1.set(ylabel="mm/day", ylim=(0, 120)); a1.legend(fontsize=7)
fig.suptitle("Out-of-fold forecasts, fold 5 (target dates Mar-Oct 2018), Set B"); fig.tight_layout()
save(fig, "04_oof_sanity_fold5.png"); plt.show()

<<A8>>

### Step 10 — What the coefficients say: top standardised coefficients

**WHAT:** fit `logreg_l2` and `ridge` on all development rows (targets 1981–2019; no test data), Set B, and plot the 15
largest standardised coefficients (change in log-odds or mm per 1-std change of the feature, holding the others fixed).

**WHY:** linear models are interpretable — but only *conditionally*: with many correlated features (QV2M, T2MDEW, their lags
and means) coefficients share credit and can even flip sign. Read them as "what the model uses", not as causal effects.

**Alternatives considered:** permutation importance (model-agnostic) — will be used for all models in a later stage.

**Look for:** whether the dominant terms are rain persistence, moisture, pressure tendency and season, and any
counter-intuitive signs that reveal collinearity.

In [ ]:
X, Y = E.load("B"); dev = S.dev_index(X.index)
def fitted_coefs(factory, target):
    pipe = factory(0, Y.loc[dev, target]).fit(X.loc[dev], Y.loc[dev, target])
    names = [n.split("__", 1)[1] for n in pipe[0].get_feature_names_out()]
    return pd.Series(np.ravel(pipe[-1].coef_), index=names), pipe

coef_c, _ = fitted_coefs(Mo.LINEAR_CLS["logreg_l2"][0], "y_cls")
coef_r, pipe_r = fitted_coefs(Mo.LINEAR_REG["ridge"][0], "y_reg")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, c, title in ((axes[0], coef_c, "logreg_l2: log-odds per 1 std"), (axes[1], coef_r, "ridge: mm per 1 std")):
    top = c.reindex(c.abs().sort_values(ascending=False).index[:15])[::-1]
    ax.barh(top.index, top, color=np.where(top > 0, "tab:blue", "tab:red")); ax.axvline(0, color="k", lw=0.8)
    ax.set_title(title, fontsize=10); ax.tick_params(labelsize=8)
fig.suptitle("Top-15 standardised coefficients (Set B, fitted on 1981-2019)"); fig.tight_layout()
save(fig, "04_linear_top_coefficients.png"); plt.show()
pd.DataFrame({"logreg_l2": coef_c, "ridge": coef_r}).round(4).to_csv(TAB / "04_linear_coefficients_setB.csv")

<<A9>>

### Step 11 — Coefficient paths and the features Lasso / L1-logistic keep

**WHAT:** (a) the Lasso path for regression (`lasso_path`, standardised inputs, 40 penalties); (b) the L1-logistic path
(8 values of C). (c) The features with non-zero coefficients at the penalties used in Step 6 (Lasso α = 0.1, L1-logistic
C = 0.1), fitted on the development rows.

**WHY:** a path shows the *order* in which features enter as the penalty is relaxed — the earliest ones carry the most
unique information. The selected set shows how much of the 115-feature table a linear model really needs.

**Alternatives considered:** stability selection (refit on bootstrap samples) — more robust, more compute; later if needed.

**Look for:** the first ~5 features to enter, and how many features survive.

In [ ]:
pre = Mo.linear_pre().fit(X.loc[dev])
names = [n.split("__", 1)[1] for n in pre.get_feature_names_out()]
Z = pre.transform(X.loc[dev]); Zc = Z - Z.mean(0)
yr = Y.loc[dev, "y_reg"].to_numpy(); yc = Y.loc[dev, "y_cls"].to_numpy()
alphas, path_r, _ = lasso_path(Zc, yr - yr.mean(), alphas=np.logspace(1, -2.5, 40))
Cs = np.logspace(-3, 0, 8)
path_c = np.array([LogisticRegression(C=C, l1_ratio=1.0, solver="saga", max_iter=5000, random_state=0)
                   .fit(Z, yc).coef_.ravel() for C in Cs]).T

def entry_order(path):           # index of the first penalty at which each feature becomes non-zero
    nz = np.abs(path) > 1e-8
    return pd.Series(np.where(nz.any(1), nz.argmax(1), np.inf), index=names).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, path, x, lab, title in ((axes[0], path_r, alphas, "alpha (log)", "Lasso path (regression, mm)"),
                                (axes[1], path_c, Cs, "C (log)", "L1-logistic path (rain >= 1 mm)")):
    first = entry_order(path).index[:6]
    for i, n in enumerate(names):
        ax.plot(x, path[i], lw=1.4 if n in first else 0.4, color=None if n in first else "lightgrey",
                label=n if n in first else None)
    ax.set_xscale("log"); ax.set(xlabel=lab, ylabel="coefficient", title=title); ax.legend(fontsize=7)
axes[0].invert_xaxis()
fig.tight_layout(); save(fig, "04_linear_coefficient_paths.png"); plt.show()

lasso_fit = Mo.LINEAR_REG["lasso"][0](0, None).fit(X.loc[dev], yr)
l1_fit = Mo.LINEAR_CLS["logreg_l1"][0](0, None).fit(X.loc[dev], yc)
sel = {m: pd.Series(np.ravel(p[-1].coef_), index=names) for m, p in (("lasso (alpha=0.1)", lasso_fit), ("logreg_l1 (C=0.1)", l1_fit))}
sel_tab = pd.DataFrame(sel)
sel_tab[(sel_tab.abs() > 1e-8).any(axis=1)].round(4).to_csv(TAB / "04_l1_selected_features_setB.csv")
print({m: f"{(s.abs() > 1e-8).sum()} of {len(s)} features kept" for m, s in sel.items()})
print("first 10 to enter - Lasso:", list(entry_order(path_r).index[:10]))
print("first 10 to enter - L1-logistic:", list(entry_order(path_c).index[:10]))

<<A10>>

<<WHYLINEAR>>

# Part 3 — Prompt 4.3: tree ensembles and gradient boosting

### Step 12 — Bagging vs boosting, scaling, and the hyper-parameters we use

**Bagging (Random Forest).** Many deep trees, each grown on a bootstrap sample of the rows and a random subset of
features at each split; predictions are averaged. Averaging many *different*, individually over-fitted trees reduces
**variance**. More trees never over-fit (they only stabilise the average), so no early stopping is needed.

**Boosting (XGBoost, LightGBM, CatBoost).** Trees are added *sequentially*; each new small tree fits the remaining errors
(the gradient of the loss) of the current ensemble, scaled by a learning rate. This reduces **bias** step by step, but
keeps fitting noise if it runs too long → the number of trees must be controlled: **early stopping** on the
chronological slice (Step 2 design).

**Why trees need no scaling.** A split asks "is QV2M > 18.3?". Any monotone rescaling (standardising, log) moves the
cut-point but produces exactly the same partition of days, so the model is unchanged. (One-hot encoding of `month` is also
unnecessary: trees can split on the integer month directly.)

**Missing values.** XGBoost, LightGBM and CatBoost learn a default direction for missing values at each split; scikit-learn's
Random Forest also supports NaN in recent versions. Feature sets A and B contain **no NaN** (Stage 2), so this capability is
not exercised here; it matters for the radiation ablation (1981–1983 NaN) in Stage 6.

**Class weighting.** RF: `class_weight="balanced"`; XGBoost: `scale_pos_weight = n_dry / n_wet` of the training rows;
LightGBM: `class_weight="balanced"`; CatBoost: `auto_class_weights="Balanced"`.

| library | key hyper-parameters (our defaults) | what they control |
|---|---|---|
| Random Forest | `n_estimators=300`, `min_samples_leaf=5`, `max_features="sqrt"` (cls) / `0.33` (reg) | number of trees; leaf size (smoothness); feature subsampling (de-correlates trees) |
| XGBoost | `learning_rate=0.03`, `max_depth=5`, `min_child_weight=5`, `subsample=0.8`, `colsample_bytree=0.8`, `reg_lambda=1`, `tree_method="hist"`, ≤ 2000 trees + early stopping | step size; tree depth (interaction order); minimum leaf weight; row/column subsampling; L2 on leaf values |
| LightGBM | `learning_rate=0.03`, `num_leaves=31`, `min_child_samples=20`, `subsample=0.8` (`subsample_freq=1`), `colsample_bytree=0.8` | grows trees **leaf-wise** (best leaf first), so `num_leaves` (not depth) controls complexity |
| CatBoost | `learning_rate=0.05`, `depth=6`, ≤ 2000 iterations + early stopping | builds **symmetric (oblivious)** trees and uses ordered boosting, which reduces over-fitting on small data |

All four are stochastic (row/feature subsampling, bootstrap) → **5 seeds** each. Early stopping uses the libraries' default
loss (log loss / RMSE) on the slice; the Stage-6 tuning will use the pre-registered metrics (PR-AUC, MAE).

**Look for (next step):** the number of trees chosen by early stopping, and run times.

In [ ]:
run_family({k: (f, True) for k, f in Mo.trees("cls").items()}, "cls")
run_family({k: (f, True) for k, f in Mo.trees("reg").items()}, "reg")

<<A11>>

### Step 13 — Trees vs linear models vs baselines

**WHAT:** the same comparison table as Step 7, now with all models; then the early-stopping tree counts.

**WHY:** P1/P3 — one table answers whether non-linear models add skill beyond the linear ones and beyond season +
persistence, and whether that holds before and after 2016.

**Look for:** the size of the gains (and whether they exceed the fold std and seed std), and rule-11 hits.

In [ ]:
ALL_CLS = BASE + list(Mo.LINEAR_CLS) + list(Mo.trees("cls"))
ALL_REG = BASE + list(Mo.LINEAR_REG) + list(Mo.trees("reg"))
display(compare("cls", ALL_CLS, CLS_KEY, "BSS", "04_part1_all_models_cls"))
display(compare("reg", ALL_REG, REG_KEY, "MAE", "04_part1_all_models_reg"))
cv = pd.read_csv(E.RESULTS)
trees_n = (cv[(cv.metric == "n_trees") & cv.model.isin(["xgboost", "lightgbm", "catboost"])]
           .groupby(["model", "task", "feature_set"])["value"].agg(["mean", "min", "max"]).round(0))
display(trees_n)
rule11()

<<A12>>

### Step 14 — Figure: headline skill of every model (Set A vs B, folds 1–3 vs 4–5)

**WHAT:** BSS vs climatology and BSS vs seasonal + persistence (classification), MAE and MSESS vs seasonal + persistence
(regression); bars = 5-fold means for Set B, dots = Set A; the dashed line marks the seasonal + persistence reference.

**WHY:** the one-glance summary for the review before Stage 4 part 2.

**Look for:** how far above the dashed line each family sits, and A vs B.

In [ ]:
cv = pd.read_csv(E.RESULTS)
panels = [("cls", "BSS"), ("cls", "BSS_vs_seas_pers"), ("reg", "MAE"), ("reg", "MSESS_vs_seas_pers")]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, (task, met) in zip(axes, panels):
    models = [m for m in (ALL_CLS if task == "cls" else ALL_REG) if not m.startswith("baseline")]
    s = E.summarise(cv[(cv.task == task) & cv.model.isin(models)], [met]).set_index(["model", "feature_set"])
    xb = [s.loc[(m, "B"), "mean"] for m in models]; eb = [s.loc[(m, "B"), "fold_std"] for m in models]
    ax.bar(range(len(models)), xb, yerr=eb, capsize=2, color=["tab:blue"] * 5 + ["tab:green"] * 4, alpha=0.8, label="Set B")
    ax.plot(range(len(models)), [s.loc[(m, "A"), "mean"] for m in models], "ko", ms=4, label="Set A")
    ref = cv[(cv.model == "baseline:seasonal+persistence") & (cv.task == task) & (cv.metric == met)]["value"].mean()
    ax.axhline(ref, color="tab:red", ls="--", lw=1, label="seasonal+persistence")
    ax.set_xticks(range(len(models)), models, rotation=70, fontsize=7); ax.set_title(met, fontsize=10)
    if met == "MAE": ax.set_ylim(5, 9)
axes[0].legend(fontsize=7)
fig.suptitle("Stage 4 part 1: 5-fold means (bars: Set B ± fold std, dots: Set A); blue = linear, green = trees")
fig.tight_layout(); save(fig, "04_part1_model_comparison.png"); plt.show()

<<A13>>

<<SUMMARY>>